# Plant Disease Classification — Training (Colab)

This notebook runs the full training pipeline on Colab's free GPU.

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Upload the dataset to Google Drive at `MyDrive/plant_disease_data/raw/{train,val}/`
3. Have a Weights & Biases account ready (free: https://wandb.ai)

## 1. Setup: clone repo, mount Drive, install deps

In [ ]:
# Check GPU
!nvidia-smi | head -20

In [ ]:
import os
REPO_URL = "https://github.com/Stepang08/Plant_Disease_Detection.git"
if not os.path.exists("/content/Plant_Disease_Detection"):
    !git clone {REPO_URL}
%cd /content/Plant_Disease_Detection

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Symlink the dataset from Drive into the project layout the code expects.
# Dataset lives at: MyDrive/plant_disease_data/PlantDoc/{train,val}
DRIVE_DATA_ROOT = "/content/drive/MyDrive/plant_disease_data/PlantDoc"
!mkdir -p data/raw data/processed
!rm -rf data/raw/train data/raw/val
!ln -s "{DRIVE_DATA_ROOT}/train" data/raw/train
!ln -s "{DRIVE_DATA_ROOT}/val" data/raw/val
!ls data/raw/train | head -5 && echo "..." && ls data/raw/train | wc -l

In [ ]:
# Install deps. Colab has torch preinstalled; only need the lighter packages.
!pip install -q timm wandb scikit-learn pyyaml

## 2. Generate label mapping and train/val split

In [ ]:
!python -m src.dataset

## 3. Weights & Biases login

In [ ]:
import wandb
wandb.login()  # paste API key from https://wandb.ai/authorize

## 4. Smoke test (optional, ~1 min)

Quick sanity check before committing to a full training run.

In [ ]:
!python -m src.train --config configs/default.yaml --smoke-test --no-wandb

## 5. Full training

Checkpoints save to `models/checkpoints/latest.pth` and `models/best_model.pth`.
We also back them up to Drive so they survive Colab session disconnects.

In [ ]:
# Make the checkpoint dir a symlink to Drive so saves are persistent.
!mkdir -p /content/drive/MyDrive/plant_disease_data/checkpoints
!rm -rf models/checkpoints
!ln -s /content/drive/MyDrive/plant_disease_data/checkpoints models/checkpoints

In [ ]:
!python -m src.train --config configs/default.yaml

## 6. Resume from checkpoint (if session died)

Resume support lives in `src/train.py`. To resume, copy the checkpoint into place and re-run — future work.

## 7. Ablation runs

Each ablation is a one-field YAML override. Example: try ConvNeXt-Tiny.

In [ ]:
# Quick hack: sed replace the backbone line in a copy of the config.
!cp configs/default.yaml configs/convnext_tiny.yaml
!sed -i 's/backbone: efficientnet_b0/backbone: convnext_tiny/' configs/convnext_tiny.yaml
!python -m src.train --config configs/convnext_tiny.yaml